# ⛳ Download Data Module

## How to use
| Step | Cell | When to run |
|------|------|-------------|
| 1 | **Install** | Once per session |
| 1.5 | **Mount Google Drive** | Once per Colab session — shows Google authorization prompt |
| 2 | **SGX Download** | Once — saves data to `/content`, then asks you to save locally |
| 3 | **S&P 500 Download** | Once — saves data to `/content`, then asks you to save locally |
| 4 | **S&P 500 Scanner** | Anytime — loads from memory or CSV |
| 5 | **S&P 500 RSI Scanner** | Anytime — loads from memory or CSV |
| 7 | **EX Scanner (SGX/S&P 500)** | After downloading data |
| 8 | **MS Scanner (SGX/S&P 500)** | After downloading data |
| 9 | **AM Scanner (SGX/S&P 500)** | After downloading data |
| 10 | **RB Scanner (SGX + S&P 500)** | After downloading data |
| 11 | **Combine & Download** | Run last to get detailed CSV + TradingView watchlist |

> **Tip:** After running a Download cell, all scanner cells below it  
> will reuse the data already in memory — no re-download needed.

> **Drive backup:** Drive backup now runs near Cell 11. It empties `ScannerData` only if this session created fresh online history files in `/content`, then copies those files into Drive.

> **Fallback:** Download cells now auto-initialize fallback helpers and attempt Google Drive mount before going online.

> **Important:** Run the Mount Google Drive cell before Cell 2 if you want the notebook to use files from `MyDrive/ScannerData`.


In [1]:
from IPython.display import display, HTML, clear_output


def update_progress(step_name, progress_pct):
    """Manually update the progress bar in Colab."""
    html = f"""
    <div style="width: 100%; border: 1px solid #ccc; border-radius: 5px; background: #f0f0f0;">
        <div style="width: {progress_pct}%; background: #4CAF50; color: white; text-align: center; border-radius: 5px; padding: 5px 0;">
            {step_name} ({progress_pct}%)
        </div>
    </div>
    """
    clear_output(wait=True)
    display(HTML(html))


def download_to_local(file_paths):
    """Ask the browser to save one or more /content files to the local drive."""
    try:
        from google.colab import files
    except Exception:
        print("Local download is only available when running inside Google Colab.")
        return

    if isinstance(file_paths, str):
        file_paths = [file_paths]

    import os
    for file_path in file_paths:
        if os.path.exists(file_path):
            print(f"Downloading {file_path} - choose where to save it on your local drive if your browser asks.")
            files.download(file_path)
        else:
            print(f"Skipped download, file not found: {file_path}")


def _init_scanner_results():
    global scanner_results
    if "scanner_results" not in globals() or not isinstance(scanner_results, list):
        scanner_results = []
    return scanner_results


def record_scanner_pick(symbol, market, scanner, signal=None, name="", sector="", reason="", last_close=None, last_date=None):
    """Store one scanner result for Cell 11 to consolidate later."""
    rows = _init_scanner_results()
    tv_symbol = f"SGX:{symbol}" if market == "SGX" and not str(symbol).startswith("SGX:") else str(symbol)
    rows.append({
        "tv_symbol": tv_symbol,
        "symbol": str(symbol).replace("SGX:", ""),
        "market": market,
        "name": name,
        "sector": sector,
        "scanner": scanner,
        "signal": signal or scanner,
        "reason": reason or signal or scanner,
        "last_close": last_close,
        "last_date": last_date,
    })


def record_scanner_rows(rows, scanner, market):
    """Store rows produced by a scanner for Cell 11."""
    for r in rows:
        symbol = r.get("symbol") or r.get("ticker") or r.get("tv_symbol")
        if not symbol:
            continue
        record_scanner_pick(
            symbol=symbol,
            market=market,
            scanner=scanner,
            signal=r.get("signal") or scanner,
            name=r.get("name", ""),
            sector=r.get("sector", r.get("market", "")),
            reason=r.get("reason", r.get("signal", scanner)),
            last_close=r.get("last_close"),
            last_date=r.get("last_date"),
        )


# ============================================================
# Fresh historical-data loader for download/scanner cells
# ============================================================
FRESH_FILE_MAX_AGE_DAYS = 0  # 0 = only files modified today are accepted
SCANNER_DRIVE_DIR = "/content/drive/MyDrive/ScannerData"


def mount_google_drive_if_available():
    """Optional: mount Google Drive in Colab so saved CSVs can be reused."""
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Google Drive mounted at /content/drive")
        return True
    except Exception as e:
        print(f"Google Drive mount not available here: {e}")
        return False


def empty_drive_scanner_folder():
    """Empty Google Drive ScannerData before saving fresh history files."""
    import os
    import shutil

    drive_root = "/content/drive/MyDrive"
    if not os.path.isdir(drive_root):
        print("Google Drive is not mounted. ScannerData folder was not emptied.")
        return False

    os.makedirs(SCANNER_DRIVE_DIR, exist_ok=True)
    for name in os.listdir(SCANNER_DRIVE_DIR):
        path = os.path.join(SCANNER_DRIVE_DIR, name)
        try:
            if os.path.isfile(path) or os.path.islink(path):
                os.remove(path)
            elif os.path.isdir(path):
                shutil.rmtree(path)
        except Exception as e:
            print(f"Could not remove {path}: {e}")

    print(f"Emptied Google Drive folder: {SCANNER_DRIVE_DIR}")
    return True


def copy_files_to_drive_if_available(file_paths, empty_first=True):
    """Copy freshly downloaded CSVs to Google Drive ScannerData when Drive is mounted."""
    import os
    import shutil

    if isinstance(file_paths, str):
        file_paths = [file_paths]

    drive_root = "/content/drive/MyDrive"
    if not os.path.isdir(drive_root):
        print("Google Drive is not mounted, so no Drive backup copy was made.")
        return

    global _scanner_drive_folder_emptied_this_session
    if empty_first and not globals().get("_scanner_drive_folder_emptied_this_session", False):
        empty_drive_scanner_folder()
        _scanner_drive_folder_emptied_this_session = True
    else:
        os.makedirs(SCANNER_DRIVE_DIR, exist_ok=True)

    for file_path in file_paths:
        if os.path.exists(file_path):
            dest = os.path.join(SCANNER_DRIVE_DIR, os.path.basename(file_path))
            shutil.copy2(file_path, dest)
            print(f"Copied fresh file to Drive: {dest}")
        else:
            print(f"Drive copy skipped, file not found: {file_path}")


def backup_fresh_history_to_drive_near_final_step():
    """Near-final backup: empty Drive ScannerData only after fresh /content history exists."""
    import os

    fresh_files = []
    if globals().get("_sgx_history_downloaded_online_this_session", False):
        fresh_files.extend(["/content/sgx_historical.csv", "/content/sti_close.csv"])
    if globals().get("_sp500_history_downloaded_online_this_session", False):
        fresh_files.extend(["/content/sp500_historical.csv", "/content/spy_close.csv"])

    fresh_files = [f for f in fresh_files if os.path.exists(f)]
    if not fresh_files:
        print("No fresh online history files were created this session, so Drive ScannerData was not emptied.")
        return

    if "copy_files_to_drive_if_available" in globals():
        copy_files_to_drive_if_available(fresh_files, empty_first=True)
    else:
        print("Drive backup helper is not available.")


def _scanner_candidate_dirs():
    """Folders to search for today's saved historical CSV files."""
    import os
    from pathlib import Path

    candidates = []
    env_dir = os.environ.get("SCANNER_DOWNLOAD_DIR")
    if env_dir:
        candidates.append(env_dir)

    candidates.extend([
        "/content",
        SCANNER_DRIVE_DIR,
        str(Path.home() / "Downloads"),
        "/Users/ky/Downloads",
    ])

    seen, existing = set(), []
    for folder in candidates:
        if folder and folder not in seen:
            seen.add(folder)
            if os.path.isdir(folder):
                existing.append(folder)
    return existing


def _local_file_is_fresh(file_path, max_age_days=FRESH_FILE_MAX_AGE_DAYS):
    """Return True only for saved files considered fresh enough to scan."""
    import os
    from datetime import datetime, timedelta

    if not file_path or not os.path.exists(file_path):
        return False

    modified_time = datetime.fromtimestamp(os.path.getmtime(file_path))
    if max_age_days == 0:
        return modified_time.date() == datetime.now().date()
    return modified_time >= datetime.now() - timedelta(days=max_age_days)


def _latest_file(patterns, fresh_only=True):
    """Find the newest matching file, optionally requiring it to be fresh."""
    import glob
    import os

    matches = []
    for folder in _scanner_candidate_dirs():
        for pattern in patterns:
            matches.extend(glob.glob(os.path.join(folder, pattern)))

    matches = [m for m in matches if os.path.isfile(m)]
    if fresh_only:
        stale = [m for m in matches if not _local_file_is_fresh(m)]
        matches = [m for m in matches if _local_file_is_fresh(m)]
        for file_path in sorted(stale, key=os.path.getmtime, reverse=True)[:3]:
            print(f"Ignoring stale file: {file_path}")

    if not matches:
        return None
    return max(matches, key=os.path.getmtime)


def _load_sgx_history_from_files(hist_path, sti_path):
    """Load SGX historical data and STI benchmark into notebook memory."""
    global sgx_stock_data, sgx_ticker_info, sti_close
    import pandas as pd

    df = pd.read_csv(hist_path, parse_dates=["date"])
    sgx_ticker_info = {}
    sgx_stock_data = {}
    for sym, grp in df.groupby("symbol"):
        yf_t = f"{sym}.SI"
        grp = grp.set_index("date").sort_index()
        sgx_stock_data[yf_t] = grp[["open", "high", "low", "close", "volume"]]
        sgx_ticker_info[yf_t] = {
            "symbol": sym,
            "name": grp["name"].iloc[0] if "name" in grp else "",
            "market": grp["market"].iloc[0] if "market" in grp else "",
        }
    sti_close = pd.read_csv(sti_path, index_col=0, parse_dates=True).squeeze()
    print(f"Loaded fresh SGX historical file: {hist_path}")
    print(f"Loaded fresh STI benchmark file : {sti_path}")
    print(f"SGX stocks loaded              : {len(sgx_stock_data)}")
    return True


def _load_sp500_history_from_files(hist_path, spy_path):
    """Load S&P 500 historical data and SPY benchmark into notebook memory."""
    global sp500_stock_data, sp500_ticker_info, spy_close
    import pandas as pd

    df = pd.read_csv(hist_path, parse_dates=["date"])
    sp500_ticker_info = {}
    sp500_stock_data = {}
    for sym, grp in df.groupby("symbol"):
        yf_t = str(sym).replace(".", "-")
        grp = grp.set_index("date").sort_index()
        sp500_stock_data[yf_t] = grp[["open", "high", "low", "close", "volume"]]
        sp500_ticker_info[yf_t] = {
            "symbol": sym,
            "name": grp["name"].iloc[0] if "name" in grp else "",
            "sector": grp["sector"].iloc[0] if "sector" in grp else "",
        }
    spy_close = pd.read_csv(spy_path, index_col=0, parse_dates=True).squeeze()
    print(f"Loaded fresh S&P 500 historical file: {hist_path}")
    print(f"Loaded fresh SPY benchmark file      : {spy_path}")
    print(f"S&P 500 stocks loaded               : {len(sp500_stock_data)}")
    return True


def ensure_scanner_data_loaded(require_sgx=True, require_sp500=True, fresh_only=True):
    """Use memory first, then fresh saved files from /content, Drive, or Downloads."""
    sgx_ok = ("sgx_stock_data" in globals() and "sgx_ticker_info" in globals() and "sti_close" in globals())
    sp500_ok = ("sp500_stock_data" in globals() and "sp500_ticker_info" in globals() and "spy_close" in globals())

    if require_sgx and not sgx_ok:
        sgx_hist = _latest_file(["sgx_historical*.csv"], fresh_only=fresh_only)
        sti_file = _latest_file(["sti_close*.csv"], fresh_only=fresh_only)
        if sgx_hist and sti_file:
            try:
                sgx_ok = _load_sgx_history_from_files(sgx_hist, sti_file)
            except Exception as e:
                print(f"Could not load SGX data from saved files: {e}")
                sgx_ok = False
        else:
            print("No fresh SGX saved files found. Online download is needed, or upload today's sgx_historical.csv and sti_close.csv.")

    if require_sp500 and not sp500_ok:
        sp500_hist = _latest_file(["sp500_historical*.csv"], fresh_only=fresh_only)
        spy_file = _latest_file(["spy_close*.csv"], fresh_only=fresh_only)
        if sp500_hist and spy_file:
            try:
                sp500_ok = _load_sp500_history_from_files(sp500_hist, spy_file)
            except Exception as e:
                print(f"Could not load S&P 500 data from saved files: {e}")
                sp500_ok = False
        else:
            print("No fresh S&P 500 saved files found. Online download is needed, or upload today's sp500_historical.csv and spy_close.csv.")

    if not _scanner_candidate_dirs():
        print("No readable local folder is visible. In Colab, upload files to /content or mount Google Drive first.")

    return {"sgx": sgx_ok, "sp500": sp500_ok}


update_progress("Ready", 0)


## Cell 1 — Install
Run once per Colab session.

In [ ]:
!pip install yfinance requests lxml -q

---
## Cell 1.5 — Mount Google Drive
Run once per Colab session. This makes `/content/drive/MyDrive/ScannerData` visible before the download/scanner cells check for fresh historical CSV files.


In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE - run once per Colab session
# ============================================================
import os

SCANNER_DRIVE_DIR = "/content/drive/MyDrive/ScannerData"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(SCANNER_DRIVE_DIR, exist_ok=True)
    print(f"Google Drive ready. ScannerData folder: {SCANNER_DRIVE_DIR}")
except Exception as e:
    print(f"Google Drive mount is only available in Colab, or it was already mounted. Details: {e}")


---
## Cell 2 — SGX Download
Uses fresh SGX CSVs first. If today's `sgx_historical*.csv` and `sti_close*.csv` are found in `/content`, Google Drive `ScannerData`, or Downloads, it loads them and skips online download. If they are missing or stale, it downloads online.


In [ ]:
# ============================================================
# SGX DOWNLOAD — saves sgx_historical.csv + sti_close.csv
# Run once per session. Scanner cells load from these files.
# ============================================================
import csv, time, warnings
import requests, pandas as pd, yfinance as yf
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

SGX_HIST_CSV = "/content/sgx_historical.csv"
STI_CSV      = "/content/sti_close.csv"
BATCH_SIZE   = 50
BENCHMARK    = "^STI"
MIN_VOLUME   = 100_000
MIN_PRICE    = 0.20

print("=" * 55)
print("  SGX DATA DOWNLOAD")
print("=" * 55)


# Bootstrap fallback helpers if the setup/progress cell was not run.
if "ensure_scanner_data_loaded" not in globals():
    print("Setup/progress helper was not run yet. Initializing fallback helpers now...")
    exec('# ============================================================\n# Fresh historical-data loader for download/scanner cells\n# ============================================================\nFRESH_FILE_MAX_AGE_DAYS = 0  # 0 = only files modified today are accepted\nSCANNER_DRIVE_DIR = "/content/drive/MyDrive/ScannerData"\n\n\ndef mount_google_drive_if_available():\n    """Optional: mount Google Drive in Colab so saved CSVs can be reused."""\n    try:\n        from google.colab import drive\n        drive.mount("/content/drive")\n        print("Google Drive mounted at /content/drive")\n        return True\n    except Exception as e:\n        print(f"Google Drive mount not available here: {e}")\n        return False\n\n\ndef empty_drive_scanner_folder():\n    """Empty Google Drive ScannerData before saving fresh history files."""\n    import os\n    import shutil\n\n    drive_root = "/content/drive/MyDrive"\n    if not os.path.isdir(drive_root):\n        print("Google Drive is not mounted. ScannerData folder was not emptied.")\n        return False\n\n    os.makedirs(SCANNER_DRIVE_DIR, exist_ok=True)\n    for name in os.listdir(SCANNER_DRIVE_DIR):\n        path = os.path.join(SCANNER_DRIVE_DIR, name)\n        try:\n            if os.path.isfile(path) or os.path.islink(path):\n                os.remove(path)\n            elif os.path.isdir(path):\n                shutil.rmtree(path)\n        except Exception as e:\n            print(f"Could not remove {path}: {e}")\n\n    print(f"Emptied Google Drive folder: {SCANNER_DRIVE_DIR}")\n    return True\n\n\ndef copy_files_to_drive_if_available(file_paths, empty_first=True):\n    """Copy freshly downloaded CSVs to Google Drive ScannerData when Drive is mounted."""\n    import os\n    import shutil\n\n    if isinstance(file_paths, str):\n        file_paths = [file_paths]\n\n    drive_root = "/content/drive/MyDrive"\n    if not os.path.isdir(drive_root):\n        print("Google Drive is not mounted, so no Drive backup copy was made.")\n        return\n\n    global _scanner_drive_folder_emptied_this_session\n    if empty_first and not globals().get("_scanner_drive_folder_emptied_this_session", False):\n        empty_drive_scanner_folder()\n        _scanner_drive_folder_emptied_this_session = True\n    else:\n        os.makedirs(SCANNER_DRIVE_DIR, exist_ok=True)\n\n    for file_path in file_paths:\n        if os.path.exists(file_path):\n            dest = os.path.join(SCANNER_DRIVE_DIR, os.path.basename(file_path))\n            shutil.copy2(file_path, dest)\n            print(f"Copied fresh file to Drive: {dest}")\n        else:\n            print(f"Drive copy skipped, file not found: {file_path}")\n\n\ndef backup_fresh_history_to_drive_near_final_step():\n    """Near-final backup: empty Drive ScannerData only after fresh /content history exists."""\n    import os\n\n    fresh_files = []\n    if globals().get("_sgx_history_downloaded_online_this_session", False):\n        fresh_files.extend(["/content/sgx_historical.csv", "/content/sti_close.csv"])\n    if globals().get("_sp500_history_downloaded_online_this_session", False):\n        fresh_files.extend(["/content/sp500_historical.csv", "/content/spy_close.csv"])\n\n    fresh_files = [f for f in fresh_files if os.path.exists(f)]\n    if not fresh_files:\n        print("No fresh online history files were created this session, so Drive ScannerData was not emptied.")\n        return\n\n    if "copy_files_to_drive_if_available" in globals():\n        copy_files_to_drive_if_available(fresh_files, empty_first=True)\n    else:\n        print("Drive backup helper is not available.")\n\n\ndef _scanner_candidate_dirs():\n    """Folders to search for today\'s saved historical CSV files."""\n    import os\n    from pathlib import Path\n\n    candidates = []\n    env_dir = os.environ.get("SCANNER_DOWNLOAD_DIR")\n    if env_dir:\n        candidates.append(env_dir)\n\n    candidates.extend([\n        "/content",\n        SCANNER_DRIVE_DIR,\n        str(Path.home() / "Downloads"),\n        "/Users/ky/Downloads",\n    ])\n\n    seen, existing = set(), []\n    for folder in candidates:\n        if folder and folder not in seen:\n            seen.add(folder)\n            if os.path.isdir(folder):\n                existing.append(folder)\n    return existing\n\n\ndef _local_file_is_fresh(file_path, max_age_days=FRESH_FILE_MAX_AGE_DAYS):\n    """Return True only for saved files considered fresh enough to scan."""\n    import os\n    from datetime import datetime, timedelta\n\n    if not file_path or not os.path.exists(file_path):\n        return False\n\n    modified_time = datetime.fromtimestamp(os.path.getmtime(file_path))\n    if max_age_days == 0:\n        return modified_time.date() == datetime.now().date()\n    return modified_time >= datetime.now() - timedelta(days=max_age_days)\n\n\ndef _latest_file(patterns, fresh_only=True):\n    """Find the newest matching file, optionally requiring it to be fresh."""\n    import glob\n    import os\n\n    matches = []\n    for folder in _scanner_candidate_dirs():\n        for pattern in patterns:\n            matches.extend(glob.glob(os.path.join(folder, pattern)))\n\n    matches = [m for m in matches if os.path.isfile(m)]\n    if fresh_only:\n        stale = [m for m in matches if not _local_file_is_fresh(m)]\n        matches = [m for m in matches if _local_file_is_fresh(m)]\n        for file_path in sorted(stale, key=os.path.getmtime, reverse=True)[:3]:\n            print(f"Ignoring stale file: {file_path}")\n\n    if not matches:\n        return None\n    return max(matches, key=os.path.getmtime)\n\n\ndef _load_sgx_history_from_files(hist_path, sti_path):\n    """Load SGX historical data and STI benchmark into notebook memory."""\n    global sgx_stock_data, sgx_ticker_info, sti_close\n    import pandas as pd\n\n    df = pd.read_csv(hist_path, parse_dates=["date"])\n    sgx_ticker_info = {}\n    sgx_stock_data = {}\n    for sym, grp in df.groupby("symbol"):\n        yf_t = f"{sym}.SI"\n        grp = grp.set_index("date").sort_index()\n        sgx_stock_data[yf_t] = grp[["open", "high", "low", "close", "volume"]]\n        sgx_ticker_info[yf_t] = {\n            "symbol": sym,\n            "name": grp["name"].iloc[0] if "name" in grp else "",\n            "market": grp["market"].iloc[0] if "market" in grp else "",\n        }\n    sti_close = pd.read_csv(sti_path, index_col=0, parse_dates=True).squeeze()\n    print(f"Loaded fresh SGX historical file: {hist_path}")\n    print(f"Loaded fresh STI benchmark file : {sti_path}")\n    print(f"SGX stocks loaded              : {len(sgx_stock_data)}")\n    return True\n\n\ndef _load_sp500_history_from_files(hist_path, spy_path):\n    """Load S&P 500 historical data and SPY benchmark into notebook memory."""\n    global sp500_stock_data, sp500_ticker_info, spy_close\n    import pandas as pd\n\n    df = pd.read_csv(hist_path, parse_dates=["date"])\n    sp500_ticker_info = {}\n    sp500_stock_data = {}\n    for sym, grp in df.groupby("symbol"):\n        yf_t = str(sym).replace(".", "-")\n        grp = grp.set_index("date").sort_index()\n        sp500_stock_data[yf_t] = grp[["open", "high", "low", "close", "volume"]]\n        sp500_ticker_info[yf_t] = {\n            "symbol": sym,\n            "name": grp["name"].iloc[0] if "name" in grp else "",\n            "sector": grp["sector"].iloc[0] if "sector" in grp else "",\n        }\n    spy_close = pd.read_csv(spy_path, index_col=0, parse_dates=True).squeeze()\n    print(f"Loaded fresh S&P 500 historical file: {hist_path}")\n    print(f"Loaded fresh SPY benchmark file      : {spy_path}")\n    print(f"S&P 500 stocks loaded               : {len(sp500_stock_data)}")\n    return True\n\n\ndef ensure_scanner_data_loaded(require_sgx=True, require_sp500=True, fresh_only=True):\n    """Use memory first, then fresh saved files from /content, Drive, or Downloads."""\n    sgx_ok = ("sgx_stock_data" in globals() and "sgx_ticker_info" in globals() and "sti_close" in globals())\n    sp500_ok = ("sp500_stock_data" in globals() and "sp500_ticker_info" in globals() and "spy_close" in globals())\n\n    if require_sgx and not sgx_ok:\n        sgx_hist = _latest_file(["sgx_historical*.csv"], fresh_only=fresh_only)\n        sti_file = _latest_file(["sti_close*.csv"], fresh_only=fresh_only)\n        if sgx_hist and sti_file:\n            try:\n                sgx_ok = _load_sgx_history_from_files(sgx_hist, sti_file)\n            except Exception as e:\n                print(f"Could not load SGX data from saved files: {e}")\n                sgx_ok = False\n        else:\n            print("No fresh SGX saved files found. Online download is needed, or upload today\'s sgx_historical.csv and sti_close.csv.")\n\n    if require_sp500 and not sp500_ok:\n        sp500_hist = _latest_file(["sp500_historical*.csv"], fresh_only=fresh_only)\n        spy_file = _latest_file(["spy_close*.csv"], fresh_only=fresh_only)\n        if sp500_hist and spy_file:\n            try:\n                sp500_ok = _load_sp500_history_from_files(sp500_hist, spy_file)\n            except Exception as e:\n                print(f"Could not load S&P 500 data from saved files: {e}")\n                sp500_ok = False\n        else:\n            print("No fresh S&P 500 saved files found. Online download is needed, or upload today\'s sp500_historical.csv and spy_close.csv.")\n\n    if not _scanner_candidate_dirs():\n        print("No readable local folder is visible. In Colab, upload files to /content or mount Google Drive first.")\n\n    return {"sgx": sgx_ok, "sp500": sp500_ok}')

# If Google Drive is not mounted yet, try to mount it before checking Drive ScannerData.
if "mount_google_drive_if_available" in globals():
    import os as _os
    if not _os.path.isdir("/content/drive/MyDrive"):
        mount_google_drive_if_available()

# Try saved historical files first so offline runs do not hang at the download step.
_sgx_loaded_from_saved_files = False
if "ensure_scanner_data_loaded" in globals():
    _status = ensure_scanner_data_loaded(require_sgx=True, require_sp500=False)
    if _status.get("sgx"):
        _sgx_loaded_from_saved_files = True
        print("Saved SGX historical data found in memory, /content, or Downloads.")
        print("Skipping online download. You can run the scanner cells now.")
else:
    print("Fallback helpers are now initialized automatically. If Drive files are not found, check that Google Drive is mounted and files are in /content/drive/MyDrive/ScannerData.")

if not _sgx_loaded_from_saved_files:
    # ── Fetch SGX stock list ─────────────────────────────────────
    print("\n[1/3] Fetching SGX stock list...")
    resp = requests.get("https://api.sgx.com/securities/v1.1/",
                        headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()
    stocks = [s for s in resp.json()["data"]["prices"] if s.get("type") == "stocks"]
    sgx_ticker_info = {
        f"{s['nc']}.SI": {
            "symbol": s["nc"], "name": s.get("n", ""), "market": s.get("m", ""),
            "sgx_pe": s.get("pe"), "sgx_pb": s.get("pb"),
            "sgx_yld": s.get("yld") or s.get("dy"),
            "sgx_eps": s.get("eps") or s.get("es"),
            "sgx_mc":  s.get("mc")  or s.get("mktcap"),
        }
        for s in stocks if s.get("nc")
    }
    print(f"    Found {len(sgx_ticker_info)} stocks")

    # ── Download price history ───────────────────────────────────
    end_date   = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
    print(f"\n[2/3] Downloading data ({start_date} → {end_date})...")

    # STI benchmark
    print(f"    Fetching benchmark {BENCHMARK}...")
    sti_raw   = yf.download(BENCHMARK, start=start_date, end=end_date, auto_adjust=True, progress=False)
    sti_close = sti_raw["Close"].squeeze()
    sti_close.to_csv(STI_CSV)

    all_tickers = list(sgx_ticker_info.keys())
    batches     = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
    sgx_stock_data = {}

    with open(SGX_HIST_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["symbol", "name", "market", "date", "open", "high", "low", "close", "volume"])
        for i, batch in enumerate(batches):
            print(f"    Batch {i+1}/{len(batches)}...", end=" ", flush=True)
            try:
                raw = yf.download(batch, start=start_date, end=end_date,
                                  auto_adjust=True, progress=False, threads=True)
                results = {}
                if len(batch) == 1:
                    if not raw.empty:
                        results[batch[0]] = raw.rename(columns=str.lower)
                else:
                    for t in batch:
                        try:
                            df = raw.xs(t, level=1, axis=1).dropna(how="all")
                            if not df.empty:
                                results[t] = df.rename(columns=str.lower)
                        except KeyError:
                            pass
                for yf_t, df in results.items():
                    if len(df) == 0: continue
                    try:
                        last_vol = float(df["volume"].iloc[-1])
                        last_price = float(df["close"].iloc[-1])
                        if last_vol <= MIN_VOLUME or last_price <= MIN_PRICE:
                            continue
                    except:
                        continue

                    info = sgx_ticker_info[yf_t]
                    sgx_stock_data[yf_t] = df

                    for date, row in df.iterrows():
                        writer.writerow([
                            info["symbol"], info["name"], info["market"],
                            date.strftime("%Y-%m-%d"),
                            round(float(row.get("open",  0) or 0), 4),
                            round(float(row.get("high",  0) or 0), 4),
                            round(float(row.get("low",   0) or 0), 4),
                            round(float(row.get("close", 0) or 0), 4),
                            int(row.get("volume", 0) or 0),
                        ])
                print(f"ok ({len(results)}/{len(batch)})")
            except Exception as e:
                print(f"ERROR: {e}")
            time.sleep(0.3)

    print(f"\n[3/3] Done!")
    print(f"    Stocks downloaded : {len(sgx_stock_data)}")
    print(f"    Saved to          : {SGX_HIST_CSV}")
    print(f"    STI benchmark     : {STI_CSV}")
    print("    ✅ Scanner cells can now use this data without re-downloading.")

    # Ask user to save downloaded data locally
    _sgx_history_downloaded_online_this_session = True
    print("Fresh online history files are ready in /content. Drive backup will run near the final step if Drive is mounted.")

    if "download_to_local" in globals():
        download_to_local([SGX_HIST_CSV, STI_CSV])
    else:
        print("Run the setup/progress cell first if you want automatic local download prompts.")


  SGX DATA DOWNLOAD

[1/3] Fetching SGX stock list...
    Found 560 stocks

[2/3] Downloading data (2025-05-22 → 2026-05-22)...
    Fetching benchmark ^STI...
    Batch 1/12... ok (50/50)
    Batch 2/12... ok (50/50)
    Batch 3/12... ok (50/50)
    Batch 4/12... ok (50/50)
    Batch 5/12... ok (50/50)
    Batch 6/12... 

ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['M12.SI', 'JCO.SI']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-05-22 -> 2026-05-22)')


ok (48/50)
    Batch 7/12... 

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['N33.SI']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-05-22 -> 2026-05-22)')


ok (49/50)
    Batch 8/12... ok (50/50)
    Batch 9/12... 

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: Y5DR.SI"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: QBPR.SI"}}}
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['Y5DR.SI', 'QBPR.SI']: YFTzMissingError('possibly delisted; no timezone found')


ok (48/50)
    Batch 10/12... ok (50/50)
    Batch 11/12... ok (50/50)
    Batch 12/12... ok (10/10)

[3/3] Done!
    Stocks downloaded : 122
    Saved to          : /content/sgx_historical.csv
    STI benchmark     : /content/sti_close.csv
    ✅ Scanner cells can now use this data without re-downloading.


---
## Cell 2b — SGX Download via Moomoo API *(alternative to Cell 2)*

Use this cell **instead of Cell 2** if you want to pull SGX data from your Moomoo account.

**One-time setup (do this on your PC/Mac before running Colab):**
1. Open Moomoo desktop → **Settings → OpenAPI → Enable OpenD** (default port 11111)
2. Install ngrok: https://ngrok.com/download
3. In a terminal on your PC/Mac run: `ngrok tcp 11111`
4. Copy the forwarding address, e.g. `tcp://0.tcp.ap.ngrok.io:12345`
5. Paste the host and port into `OPEND_HOST` / `OPEND_PORT` in the cell below

> **Same machine (not Colab)?** Keep `OPEND_HOST = "127.0.0.1"` and `OPEND_PORT = 11111`


In [ ]:
# ============================================================
# SGX DOWNLOAD via MOOMOO API  (alternative to Cell 2)
# Produces the same sgx_historical.csv + sti_close.csv as Cell 2.
# All scanner cells (6-10) work without any changes.
# ============================================================

# ── Step 0: install (run once per Colab session) ─────────────
# !pip install moomoo-api -q

# ── Step 1: configure your OpenD connection ──────────────────
OPEND_HOST = "127.0.0.1"        # Colab: paste ngrok host, e.g. "0.tcp.ap.ngrok.io"
OPEND_PORT = 11111              # Colab: paste ngrok port, e.g. 12345

# ── Step 2: run the downloader ───────────────────────────────
import importlib, sys, os

# Load the downloader from the repo (works both locally and in Colab after cloning)
_script_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '/content'
_module_path = os.path.join(_script_dir, 'moomoo_sgx_downloader.py')

if not os.path.exists(_module_path):
    # Colab: download the helper directly from your repo
    import urllib.request
    _raw_url = 'https://raw.githubusercontent.com/gfky1356-ship-it/sgx-data-download/main/moomoo_sgx_downloader.py'
    urllib.request.urlretrieve(_raw_url, _module_path)
    print(f'Downloaded moomoo_sgx_downloader.py to {_module_path}')

import importlib.util as _ilu
_spec = _ilu.spec_from_file_location('moomoo_sgx_downloader', _module_path)
_mod  = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

# Override config from this cell
_mod.OPEND_HOST = OPEND_HOST
_mod.OPEND_PORT = OPEND_PORT

sgx_stock_data, sgx_ticker_info, sti_close = _mod.download_sgx_with_moomoo(
    opend_host=OPEND_HOST,
    opend_port=OPEND_PORT,
)

# Mark data as loaded so scanner cells skip the online-download check
_sgx_history_downloaded_online_this_session = True
print('\n✅ Moomoo SGX data loaded. Scanner cells (6-10) are ready to run.')


---
## Cell 3 — S&P 500 Download
Uses fresh S&P 500 CSVs first. If today's `sp500_historical*.csv` and `spy_close*.csv` are found in `/content`, Google Drive `ScannerData`, or Downloads, it loads them and skips online download. If they are missing or stale, it downloads online.


# *** Indicator Scanner

In [ ]:
# ============================================================
# S&P 500 DOWNLOAD — saves sp500_historical.csv + spy_close.csv
# Run once per session. Scanner cells load from these files.
# ============================================================
import csv, io, time, warnings
import requests, pandas as pd, yfinance as yf
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

SP500_HIST_CSV = "/content/sp500_historical.csv"
SPY_CSV        = "/content/spy_close.csv"
BATCH_SIZE     = 50

print("=" * 55)
print("  S&P 500 DATA DOWNLOAD")
print("=" * 55)


# Bootstrap fallback helpers if the setup/progress cell was not run.
if "ensure_scanner_data_loaded" not in globals():
    print("Setup/progress helper was not run yet. Initializing fallback helpers now...")
    exec('# ============================================================\n# Fresh historical-data loader for download/scanner cells\n# ============================================================\nFRESH_FILE_MAX_AGE_DAYS = 0  # 0 = only files modified today are accepted\nSCANNER_DRIVE_DIR = "/content/drive/MyDrive/ScannerData"\n\n\ndef mount_google_drive_if_available():\n    """Optional: mount Google Drive in Colab so saved CSVs can be reused."""\n    try:\n        from google.colab import drive\n        drive.mount("/content/drive")\n        print("Google Drive mounted at /content/drive")\n        return True\n    except Exception as e:\n        print(f"Google Drive mount not available here: {e}")\n        return False\n\n\ndef empty_drive_scanner_folder():\n    """Empty Google Drive ScannerData before saving fresh history files."""\n    import os\n    import shutil\n\n    drive_root = "/content/drive/MyDrive"\n    if not os.path.isdir(drive_root):\n        print("Google Drive is not mounted. ScannerData folder was not emptied.")\n        return False\n\n    os.makedirs(SCANNER_DRIVE_DIR, exist_ok=True)\n    for name in os.listdir(SCANNER_DRIVE_DIR):\n        path = os.path.join(SCANNER_DRIVE_DIR, name)\n        try:\n            if os.path.isfile(path) or os.path.islink(path):\n                os.remove(path)\n            elif os.path.isdir(path):\n                shutil.rmtree(path)\n        except Exception as e:\n            print(f"Could not remove {path}: {e}")\n\n    print(f"Emptied Google Drive folder: {SCANNER_DRIVE_DIR}")\n    return True\n\n\ndef copy_files_to_drive_if_available(file_paths, empty_first=True):\n    """Copy freshly downloaded CSVs to Google Drive ScannerData when Drive is mounted."""\n    import os\n    import shutil\n\n    if isinstance(file_paths, str):\n        file_paths = [file_paths]\n\n    drive_root = "/content/drive/MyDrive"\n    if not os.path.isdir(drive_root):\n        print("Google Drive is not mounted, so no Drive backup copy was made.")\n        return\n\n    global _scanner_drive_folder_emptied_this_session\n    if empty_first and not globals().get("_scanner_drive_folder_emptied_this_session", False):\n        empty_drive_scanner_folder()\n        _scanner_drive_folder_emptied_this_session = True\n    else:\n        os.makedirs(SCANNER_DRIVE_DIR, exist_ok=True)\n\n    for file_path in file_paths:\n        if os.path.exists(file_path):\n            dest = os.path.join(SCANNER_DRIVE_DIR, os.path.basename(file_path))\n            shutil.copy2(file_path, dest)\n            print(f"Copied fresh file to Drive: {dest}")\n        else:\n            print(f"Drive copy skipped, file not found: {file_path}")\n\n\ndef backup_fresh_history_to_drive_near_final_step():\n    """Near-final backup: empty Drive ScannerData only after fresh /content history exists."""\n    import os\n\n    fresh_files = []\n    if globals().get("_sgx_history_downloaded_online_this_session", False):\n        fresh_files.extend(["/content/sgx_historical.csv", "/content/sti_close.csv"])\n    if globals().get("_sp500_history_downloaded_online_this_session", False):\n        fresh_files.extend(["/content/sp500_historical.csv", "/content/spy_close.csv"])\n\n    fresh_files = [f for f in fresh_files if os.path.exists(f)]\n    if not fresh_files:\n        print("No fresh online history files were created this session, so Drive ScannerData was not emptied.")\n        return\n\n    if "copy_files_to_drive_if_available" in globals():\n        copy_files_to_drive_if_available(fresh_files, empty_first=True)\n    else:\n        print("Drive backup helper is not available.")\n\n\ndef _scanner_candidate_dirs():\n    """Folders to search for today\'s saved historical CSV files."""\n    import os\n    from pathlib import Path\n\n    candidates = []\n    env_dir = os.environ.get("SCANNER_DOWNLOAD_DIR")\n    if env_dir:\n        candidates.append(env_dir)\n\n    candidates.extend([\n        "/content",\n        SCANNER_DRIVE_DIR,\n        str(Path.home() / "Downloads"),\n        "/Users/ky/Downloads",\n    ])\n\n    seen, existing = set(), []\n    for folder in candidates:\n        if folder and folder not in seen:\n            seen.add(folder)\n            if os.path.isdir(folder):\n                existing.append(folder)\n    return existing\n\n\ndef _local_file_is_fresh(file_path, max_age_days=FRESH_FILE_MAX_AGE_DAYS):\n    """Return True only for saved files considered fresh enough to scan."""\n    import os\n    from datetime import datetime, timedelta\n\n    if not file_path or not os.path.exists(file_path):\n        return False\n\n    modified_time = datetime.fromtimestamp(os.path.getmtime(file_path))\n    if max_age_days == 0:\n        return modified_time.date() == datetime.now().date()\n    return modified_time >= datetime.now() - timedelta(days=max_age_days)\n\n\ndef _latest_file(patterns, fresh_only=True):\n    """Find the newest matching file, optionally requiring it to be fresh."""\n    import glob\n    import os\n\n    matches = []\n    for folder in _scanner_candidate_dirs():\n        for pattern in patterns:\n            matches.extend(glob.glob(os.path.join(folder, pattern)))\n\n    matches = [m for m in matches if os.path.isfile(m)]\n    if fresh_only:\n        stale = [m for m in matches if not _local_file_is_fresh(m)]\n        matches = [m for m in matches if _local_file_is_fresh(m)]\n        for file_path in sorted(stale, key=os.path.getmtime, reverse=True)[:3]:\n            print(f"Ignoring stale file: {file_path}")\n\n    if not matches:\n        return None\n    return max(matches, key=os.path.getmtime)\n\n\ndef _load_sgx_history_from_files(hist_path, sti_path):\n    """Load SGX historical data and STI benchmark into notebook memory."""\n    global sgx_stock_data, sgx_ticker_info, sti_close\n    import pandas as pd\n\n    df = pd.read_csv(hist_path, parse_dates=["date"])\n    sgx_ticker_info = {}\n    sgx_stock_data = {}\n    for sym, grp in df.groupby("symbol"):\n        yf_t = f"{sym}.SI"\n        grp = grp.set_index("date").sort_index()\n        sgx_stock_data[yf_t] = grp[["open", "high", "low", "close", "volume"]]\n        sgx_ticker_info[yf_t] = {\n            "symbol": sym,\n            "name": grp["name"].iloc[0] if "name" in grp else "",\n            "market": grp["market"].iloc[0] if "market" in grp else "",\n        }\n    sti_close = pd.read_csv(sti_path, index_col=0, parse_dates=True).squeeze()\n    print(f"Loaded fresh SGX historical file: {hist_path}")\n    print(f"Loaded fresh STI benchmark file : {sti_path}")\n    print(f"SGX stocks loaded              : {len(sgx_stock_data)}")\n    return True\n\n\ndef _load_sp500_history_from_files(hist_path, spy_path):\n    """Load S&P 500 historical data and SPY benchmark into notebook memory."""\n    global sp500_stock_data, sp500_ticker_info, spy_close\n    import pandas as pd\n\n    df = pd.read_csv(hist_path, parse_dates=["date"])\n    sp500_ticker_info = {}\n    sp500_stock_data = {}\n    for sym, grp in df.groupby("symbol"):\n        yf_t = str(sym).replace(".", "-")\n        grp = grp.set_index("date").sort_index()\n        sp500_stock_data[yf_t] = grp[["open", "high", "low", "close", "volume"]]\n        sp500_ticker_info[yf_t] = {\n            "symbol": sym,\n            "name": grp["name"].iloc[0] if "name" in grp else "",\n            "sector": grp["sector"].iloc[0] if "sector" in grp else "",\n        }\n    spy_close = pd.read_csv(spy_path, index_col=0, parse_dates=True).squeeze()\n    print(f"Loaded fresh S&P 500 historical file: {hist_path}")\n    print(f"Loaded fresh SPY benchmark file      : {spy_path}")\n    print(f"S&P 500 stocks loaded               : {len(sp500_stock_data)}")\n    return True\n\n\ndef ensure_scanner_data_loaded(require_sgx=True, require_sp500=True, fresh_only=True):\n    """Use memory first, then fresh saved files from /content, Drive, or Downloads."""\n    sgx_ok = ("sgx_stock_data" in globals() and "sgx_ticker_info" in globals() and "sti_close" in globals())\n    sp500_ok = ("sp500_stock_data" in globals() and "sp500_ticker_info" in globals() and "spy_close" in globals())\n\n    if require_sgx and not sgx_ok:\n        sgx_hist = _latest_file(["sgx_historical*.csv"], fresh_only=fresh_only)\n        sti_file = _latest_file(["sti_close*.csv"], fresh_only=fresh_only)\n        if sgx_hist and sti_file:\n            try:\n                sgx_ok = _load_sgx_history_from_files(sgx_hist, sti_file)\n            except Exception as e:\n                print(f"Could not load SGX data from saved files: {e}")\n                sgx_ok = False\n        else:\n            print("No fresh SGX saved files found. Online download is needed, or upload today\'s sgx_historical.csv and sti_close.csv.")\n\n    if require_sp500 and not sp500_ok:\n        sp500_hist = _latest_file(["sp500_historical*.csv"], fresh_only=fresh_only)\n        spy_file = _latest_file(["spy_close*.csv"], fresh_only=fresh_only)\n        if sp500_hist and spy_file:\n            try:\n                sp500_ok = _load_sp500_history_from_files(sp500_hist, spy_file)\n            except Exception as e:\n                print(f"Could not load S&P 500 data from saved files: {e}")\n                sp500_ok = False\n        else:\n            print("No fresh S&P 500 saved files found. Online download is needed, or upload today\'s sp500_historical.csv and spy_close.csv.")\n\n    if not _scanner_candidate_dirs():\n        print("No readable local folder is visible. In Colab, upload files to /content or mount Google Drive first.")\n\n    return {"sgx": sgx_ok, "sp500": sp500_ok}')

# If Google Drive is not mounted yet, try to mount it before checking Drive ScannerData.
if "mount_google_drive_if_available" in globals():
    import os as _os
    if not _os.path.isdir("/content/drive/MyDrive"):
        mount_google_drive_if_available()

# Try saved historical files first so offline runs do not hang at the download step.
_sp500_loaded_from_saved_files = False
if "ensure_scanner_data_loaded" in globals():
    _status = ensure_scanner_data_loaded(require_sgx=False, require_sp500=True)
    if _status.get("sp500"):
        _sp500_loaded_from_saved_files = True
        print("Saved S&P 500 historical data found in memory, /content, or Downloads.")
        print("Skipping online download. You can run the scanner cells now.")
else:
    print("Fallback helpers are now initialized automatically. If Drive files are not found, check that Google Drive is mounted and files are in /content/drive/MyDrive/ScannerData.")

if not _sp500_loaded_from_saved_files:
    # ── Fetch ticker list from Wikipedia ────────────────────────
    print("\n[1/3] Fetching S&P 500 tickers from Wikipedia...")
    html  = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                         headers={"User-Agent": "Mozilla/5.0"}).text
    sp500 = pd.read_html(io.StringIO(html), flavor="lxml")[0]
    sp500["yf_ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
    sp500_ticker_info  = {
        row["yf_ticker"]: {"symbol": row["Symbol"], "name": row["Security"], "sector": row["GICS Sector"]}
        for _, row in sp500.iterrows()
    }
    print(f"    Found {len(sp500_ticker_info)} stocks")

    # ── Download price history ───────────────────────────────────
    end_date   = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")
    print(f"\n[2/3] Downloading data ({start_date} → {end_date})...")

    print("    Fetching SPY benchmark...")
    spy_raw   = yf.download("SPY", start=start_date, end=end_date, auto_adjust=True, progress=False)
    spy_close = spy_raw["Close"].squeeze()
    spy_close.to_csv(SPY_CSV)

    all_tickers    = list(sp500_ticker_info.keys())
    batches        = [all_tickers[i:i+BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]
    sp500_stock_data = {}

    with open(SP500_HIST_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["symbol", "name", "sector", "date", "open", "high", "low", "close", "volume"])
        for i, batch in enumerate(batches):
            print(f"    Batch {i+1}/{len(batches)}...", end=" ", flush=True)
            try:
                raw = yf.download(batch, start=start_date, end=end_date,
                                  auto_adjust=True, progress=False, threads=True)
                results = {}
                if len(batch) == 1:
                    if not raw.empty: results[batch[0]] = raw.rename(columns=str.lower)
                else:
                    for t in batch:
                        try:
                            df = raw.xs(t, level=1, axis=1).dropna(how="all")
                            if not df.empty: results[t] = df.rename(columns=str.lower)
                        except KeyError: pass
                for yf_t, df in results.items():
                    info = sp500_ticker_info[yf_t]
                    sp500_stock_data[yf_t] = df
                    for date, row in df.iterrows():
                        writer.writerow([
                            info["symbol"], info["name"], info["sector"],
                            date.strftime("%Y-%m-%d"),
                            round(float(row.get("open",  0) or 0), 4),
                            round(float(row.get("high",  0) or 0), 4),
                            round(float(row.get("low",   0) or 0), 4),
                            round(float(row.get("close", 0) or 0), 4),
                            int(row.get("volume", 0) or 0),
                        ])
                print(f"ok ({len(results)}/{len(batch)})")
            except Exception as e:
                print(f"ERROR: {e}")
            time.sleep(0.3)

    print(f"\n[3/3] Done!")
    print(f"    Stocks downloaded : {len(sp500_stock_data)}")
    print(f"    Saved to          : {SP500_HIST_CSV}")
    print(f"    SPY benchmark     : {SPY_CSV}")
    print("    ✅ Scanner cells can now use this data without re-downloading.")

    # Ask user to save downloaded data locally
    _sp500_history_downloaded_online_this_session = True
    print("Fresh online history files are ready in /content. Drive backup will run near the final step if Drive is mounted.")

    if "download_to_local" in globals():
        download_to_local([SP500_HIST_CSV, SPY_CSV])
    else:
        print("Run the setup/progress cell first if you want automatic local download prompts.")


  S&P 500 DATA DOWNLOAD

[1/3] Fetching S&P 500 tickers from Wikipedia...
    Found 503 stocks

[2/3] Downloading data (2025-05-22 → 2026-05-22)...
    Fetching SPY benchmark...
    Batch 1/11... ok (50/50)
    Batch 2/11... ok (50/50)
    Batch 3/11... ok (50/50)
    Batch 4/11... ok (50/50)
    Batch 5/11... ok (50/50)
    Batch 6/11... ok (50/50)
    Batch 7/11... ok (50/50)
    Batch 8/11... ok (50/50)
    Batch 9/11... ok (50/50)
    Batch 10/11... ok (50/50)
    Batch 11/11... ok (3/3)

[3/3] Done!
    Stocks downloaded : 503
    Saved to          : /content/sp500_historical.csv
    SPY benchmark     : /content/spy_close.csv
    ✅ Scanner cells can now use this data without re-downloading.


## Cell 5 — Data Source Check (SGX + S&P 500)
This cell does not scan. It only confirms that fresh historical data is loaded from memory, `/content`, Google Drive `ScannerData`, or Downloads. Real scanners start from Cell 6.


In [ ]:
# ============================================================
# Scanner Cell 5 - Data Source Check
# This cell does not scan. It confirms that scanner data is loaded.
# ============================================================

if "ensure_scanner_data_loaded" in globals():
    status = ensure_scanner_data_loaded()
    print("Cell 5 data source check complete.")
    print(f"SGX data ready    : {status.get('sgx')}")
    print(f"S&P 500 data ready: {status.get('sp500')}")
else:
    print("Fallback helper is not loaded. Run Cell 2 or the setup/helper cell first.")


  Scanner- CELL 5 - STOCK SCANNER  (SGX + S&P 500)

  ✅ SGX: using in-memory data.
  ✅ S&P 500: using in-memory data.

  Scanning SGX (122 stocks)...
  50/122 scanned | 3 confirmed so far
  100/122 scanned | 4 confirmed so far
  Done: 120 scanned, 5 confirmed

  Scanning S&P 500 (503 stocks)...
  50/503 scanned | 1 confirmed so far
  100/503 scanned | 1 confirmed so far
  150/503 scanned | 3 confirmed so far
  200/503 scanned | 5 confirmed so far
  250/503 scanned | 6 confirmed so far
  300/503 scanned | 7 confirmed so far
  350/503 scanned | 10 confirmed so far
  400/503 scanned | 13 confirmed so far
  450/503 scanned | 15 confirmed so far
  500/503 scanned | 16 confirmed so far
  Done: 503 scanned, 16 confirmed

  SGX - CONFIRMED SIGNALS: 5 stocks
  Symbol   Name                           Market        SMA   BO    OBV   Δ≥0 
  -------- ------------------------------ ------------ ----- ----- ----- -----
  F9D      Boustead                       MAINBOARD      Y     N     N     Y  
  4

## Scanner- Cell 6 — A+B Buy Signal + 20EXP Scanner (SGX + S&P 500)
Combines two Pine Script buy signals across SGX and S&P 500. Results are sent to Cell 11 for consolidated export.

If historical data is missing from memory, the scanner will use only fresh saved CSVs from `/content`, Google Drive `ScannerData`, or Downloads; stale files are ignored.


In [ ]:
# ============================================================
# Scanner- CELL 6 - A+B BUY SIGNAL + 20EXP SCANNER  (SGX + S&P 500)
#
# Signal 1: A+B Buy (AB buy signal.pine)
#   B: 20SMA slope > 50SMA slope
#   A: delta >= 0 two days running and today > yesterday
#   + Green candle + delta MA not going down
#
# Signal 2: 20EXP (sma20_expansion_buy.pine)
#   Part A: delta > 0, rising, > delta_MA x 1.2, MA trending, not 4th green
#   Part B: 20SMA slope > 0, > 50SMA slope, gap widening, both rising
#
# At the end: merges Cell 5 + Cell 6 into all_signals_watchlist.txt
# ============================================================
import csv as _csv, os, warnings
import pandas as pd
warnings.filterwarnings("ignore")

SGX_HIST_CSV   = "/content/sgx_historical.csv"
STI_CSV        = "/content/sti_close.csv"
SP500_HIST_CSV = "/content/sp500_historical.csv"
SPY_CSV        = "/content/spy_close.csv"
MIN_BARS       = 65
MIN_VOL_SGX    = 100_000
MIN_PRICE_SGX  = 0.20


def _sma(s, n): return s.rolling(n, min_periods=n).mean()
def _v(s, i=0):
    val = s.iloc[-(1 + i)]; return float(val) if not pd.isna(val) else float("nan")
def _nan(*vals): return any(x != x for x in vals)


def calc_ab_buy(close, open_, index_close,
                sma20_len=20, sma50_len=50, slope_lb=5, slope_sm=1,
                rs_len=30, delta_ma_len=9):
    ic   = index_close.reindex(close.index).ffill().astype(float)
    s20  = _sma(close, sma20_len);  s50 = _sma(close, sma50_len)
    sl20 = _sma((s20 - s20.shift(slope_lb)) / s20.shift(slope_lb) * 100, slope_sm)
    sl50 = _sma((s50 - s50.shift(slope_lb)) / s50.shift(slope_lb) * 100, slope_sm)
    sm   = _sma(close, rs_len);  im = _sma(ic, rs_len)
    dab  = (close / sm) - (ic / im);  dma = _sma(dab, delta_ma_len)
    sl20_0, sl50_0 = _v(sl20), _v(sl50)
    d0, d1         = _v(dab, 0), _v(dab, 1)
    dm0, dm1       = _v(dma, 0), _v(dma, 1)
    c0, o0         = _v(close), _v(open_)
    if _nan(sl20_0, sl50_0, d0, d1, dm0, dm1, c0, o0): return False, {}
    fired = (sl20_0 > sl50_0) and (d0 >= 0 and d1 >= 0 and d0 > d1) and (c0 > o0) and (dm0 >= dm1)
    return fired, {"delta_ab": round(d0, 6), "slope20": round(sl20_0, 3), "slope50": round(sl50_0, 3)}


def calc_20exp(close, index_close,
               sma_len=30, delta_ma_len=9, slope_lb=5, slope_sm=1, slope50_min=-0.05):
    ic   = index_close.reindex(close.index).ffill().astype(float)
    sm   = _sma(close, sma_len);  im = _sma(ic, sma_len)
    dab  = (close / sm) - (ic / im);  dma = _sma(dab, delta_ma_len)
    s20  = _sma(close, 20);  s50 = _sma(close, 50)
    sl20 = _sma((s20 - s20.shift(slope_lb)) / s20.shift(slope_lb) * 100, slope_sm)
    sl50 = _sma((s50 - s50.shift(slope_lb)) / s50.shift(slope_lb) * 100, slope_sm)
    gap  = sl20 - sl50
    d0,d1,d2,d3   = _v(dab,0),_v(dab,1),_v(dab,2),_v(dab,3)
    dm0,dm1       = _v(dma,0),_v(dma,1)
    sl20_0,sl20_1 = _v(sl20,0),_v(sl20,1)
    sl50_0,sl50_1 = _v(sl50,0),_v(sl50,1)
    gap0,gap1     = _v(gap,0),_v(gap,1)
    s50_0,s50_1   = _v(s50,0),_v(s50,1)
    if _nan(d0,d1,d2,d3,dm0,dm1,sl20_0,sl20_1,sl50_0,sl50_1,gap0,gap1,s50_0,s50_1): return False, {}
    part_a = d0>0 and d0>d1 and d0>dm0*1.2 and dm0>dm1 and not(d1>0 and d2>0 and d3>0)
    part_b = (sl20_0>0 and sl50_0>=slope50_min and sl20_0>sl50_0 and gap0>gap1
              and sl20_0>sl20_1 and sl50_0>sl50_1 and s50_0>s50_1)
    return part_a and part_b, {"delta_ab": round(d0,6), "slope20": round(sl20_0,3), "slope50": round(sl50_0,3)}


def scan_market(stock_data, ticker_info, index_close, market="SGX"):
    is_sgx = (market == "SGX"); results = []; total = len(stock_data)
    for i, (yf_t, df) in enumerate(stock_data.items()):
        if len(df) < MIN_BARS: continue
        if is_sgx:
            if float(df["volume"].iloc[-1]) < MIN_VOL_SGX:   continue
            if float(df["close"].iloc[-1])  < MIN_PRICE_SGX: continue
        info = ticker_info.get(yf_t, {})
        try:
            ab_fired, ab_meta   = calc_ab_buy(df["close"], df["open"], index_close)
            exp_fired, exp_meta = calc_20exp(df["close"], index_close)
        except Exception: continue
        if not (ab_fired or exp_fired): continue
        sig  = "AB+20EXP" if (ab_fired and exp_fired) else ("AB" if ab_fired else "20EXP")
        meta = ab_meta if ab_fired else exp_meta
        row  = {"symbol": info.get("symbol", yf_t), "name": info.get("name", ""),
                "signal": sig, "ab_buy": ab_fired, "20exp": exp_fired,
                "delta_ab": meta.get("delta_ab"), "slope20": meta.get("slope20"),
                "slope50": meta.get("slope50"),
                "last_close": round(float(df["close"].iloc[-1]), 4),
                "last_date":  df.index[-1].strftime("%Y-%m-%d")}
        row["market" if is_sgx else "sector"] = info.get("market" if is_sgx else "sector", "")
        results.append(row)
        if (i + 1) % 100 == 0:
            print(f"  [{market}] {i+1}/{total} scanned | hits={len(results)}")
    return results


print("=" * 70)
print("  Scanner- CELL 6 - A+B BUY + 20EXP SCANNER  (SGX + S&P 500)")
print("=" * 70)


# Load scanner data from memory, /content, or the newest matching files in Downloads.
if "ensure_scanner_data_loaded" in globals():
    ensure_scanner_data_loaded()
else:
    print("Run the setup/progress cell first so scanners can fall back to /content or Downloads files.")

# Load SGX
sgx_ok = True
try:
    _ = sgx_stock_data, sgx_ticker_info, sti_close
    print("\n  ✅ SGX: using in-memory data.")
except NameError:
    if os.path.exists(SGX_HIST_CSV) and os.path.exists(STI_CSV):
        print("\n  📂 SGX: loading from CSV...")
        _df = pd.read_csv(SGX_HIST_CSV, parse_dates=["date"])
        sgx_ticker_info = {}; sgx_stock_data = {}
        for sym, grp in _df.groupby("symbol"):
            yf_t = f"{sym}.SI"; grp = grp.set_index("date").sort_index()
            sgx_stock_data[yf_t]  = grp[["open","high","low","close","volume"]]
            sgx_ticker_info[yf_t] = {"symbol": sym, "name": grp["name"].iloc[0], "market": grp["market"].iloc[0]}
        sti_close = pd.read_csv(STI_CSV, index_col=0, parse_dates=True).squeeze()
        print(f"  Loaded {len(sgx_stock_data)} SGX stocks.")
    else:
        print("  ❌ SGX data not found - run the SGX Download cell first.")
        sgx_ok = False

# Load S&P 500
sp500_ok = True
try:
    _ = sp500_stock_data, sp500_ticker_info, spy_close
    print("  ✅ S&P 500: using in-memory data.")
except NameError:
    if os.path.exists(SP500_HIST_CSV) and os.path.exists(SPY_CSV):
        print("  📂 S&P 500: loading from CSV...")
        _df = pd.read_csv(SP500_HIST_CSV, parse_dates=["date"])
        sp500_ticker_info = {}; sp500_stock_data = {}
        for sym, grp in _df.groupby("symbol"):
            yf_t = sym.replace(".", "-"); grp = grp.set_index("date").sort_index()
            sp500_stock_data[yf_t]  = grp[["open","high","low","close","volume"]]
            sp500_ticker_info[yf_t] = {"symbol": sym, "name": grp["name"].iloc[0], "sector": grp["sector"].iloc[0]}
        spy_close = pd.read_csv(SPY_CSV, index_col=0, parse_dates=True).squeeze()
        print(f"  Loaded {len(sp500_stock_data)} S&P 500 stocks.")
    else:
        print("  ❌ S&P 500 data not found - run the S&P 500 Download cell first.")
        sp500_ok = False

sgx_ab20exp = []; sp500_ab20exp = []

if sgx_ok:
    print(f"\n  Scanning SGX ({len(sgx_stock_data)} stocks)...")
    sgx_ab20exp = scan_market(sgx_stock_data, sgx_ticker_info, sti_close, "SGX")

if sp500_ok:
    print(f"\n  Scanning S&P 500 ({len(sp500_stock_data)} stocks)...")
    sp500_ab20exp = scan_market(sp500_stock_data, sp500_ticker_info, spy_close, "SP500")

# Print SGX
_ord = {"AB+20EXP": 0, "AB": 1, "20EXP": 2}
print("\n" + "=" * 78)
print(f"  SGX - {len(sgx_ab20exp)} signals")
print("=" * 78)
if sgx_ab20exp:
    print(f"  {'Signal':<10} {'Symbol':<8} {'Name':<28} {'Market':<10} {'Close':>7} {'Δ_AB':>9} {'Sl20':>7} {'Sl50':>7}")
    print(f"  {'-'*10} {'-'*8} {'-'*28} {'-'*10} {'-'*7} {'-'*9} {'-'*7} {'-'*7}")
    for r in sorted(sgx_ab20exp, key=lambda x: (_ord.get(x["signal"],9), -(x["delta_ab"] or 0))):
        print(f"  {r['signal']:<10} {r['symbol']:<8} {r['name'][:28]:<28} {r.get('market','')[:10]:<10} "
              f"{r['last_close']:>7.4f} {r['delta_ab']:>9.5f} {r['slope20']:>+7.3f} {r['slope50']:>+7.3f}")
else:
    print("  No signals today.")

# Print S&P 500
print("\n" + "=" * 85)
print(f"  S&P 500 - {len(sp500_ab20exp)} signals")
print("=" * 85)
if sp500_ab20exp:
    print(f"  {'Signal':<10} {'Symbol':<8} {'Name':<26} {'Sector':<20} {'Close':>8} {'Δ_AB':>9} {'Sl20':>7} {'Sl50':>7}")
    print(f"  {'-'*10} {'-'*8} {'-'*26} {'-'*20} {'-'*8} {'-'*9} {'-'*7} {'-'*7}")
    for r in sorted(sp500_ab20exp, key=lambda x: (_ord.get(x["signal"],9), -(x["delta_ab"] or 0))):
        print(f"  {r['signal']:<10} {r['symbol']:<8} {r['name'][:26]:<26} {r.get('sector','')[:20]:<20} "
              f"{r['last_close']:>8.2f} {r['delta_ab']:>9.5f} {r['slope20']:>+7.3f} {r['slope50']:>+7.3f}")
else:
    print("  No signals today.")

# Send Cell 6 results to Cell 11. No separate files are exported here.
if "record_scanner_rows" in globals():
    record_scanner_rows(sgx_ab20exp, "Cell 6 - A+B / 20EXP", "SGX")
    record_scanner_rows(sp500_ab20exp, "Cell 6 - A+B / 20EXP", "S&P 500")
else:
    print("Run the setup/progress cell first so Cell 11 can consolidate results.")

print(f"\nCell 6 results sent to Cell 11: {len(sgx_ab20exp) + len(sp500_ab20exp)} picks")


  Scanner- CELL 6 - A+B BUY + 20EXP SCANNER  (SGX + S&P 500)

  ✅ SGX: using in-memory data.
  ✅ S&P 500: using in-memory data.

  Scanning SGX (122 stocks)...

  Scanning S&P 500 (503 stocks)...

  SGX - 3 signals
  Signal     Symbol   Name                         Market       Close      Δ_AB    Sl20    Sl50
  ---------- -------- ---------------------------- ---------- ------- --------- ------- -------
  AB         AWX      AEM SGD                      MAINBOARD   9.5100   0.37417 +15.216 +11.742
  AB         AP4      Riverstone                   MAINBOARD   0.9350   0.16546  +5.285  +2.812
  AB         D05      DBS                          MAINBOARD  61.7500   0.05122  +2.047  +1.374

  S&P 500 - 19 signals
  Signal     Symbol   Name                       Sector                  Close      Δ_AB    Sl20    Sl50
  ---------- -------- -------------------------- -------------------- -------- --------- ------- -------
  AB+20EXP   PRU      Prudential Financial       Financials            

## Scanner- Cell 7 — EX Scanner (极值反转)
Scans for EX-1, EX-2, and EX-4 buy signals across SGX and S&P 500.
If historical data is missing from memory, the scanner will use only fresh saved CSVs from `/content`, Google Drive `ScannerData`, or Downloads; stale files are ignored.


In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# =====================================================================
# 通用技术指标辅助函数 (Helpers)
# =====================================================================
def _sma(series, length):
    return series.rolling(window=length, min_periods=1).mean()

def _ema(series, length):
    return series.ewm(span=length, adjust=False).mean()

def _macd(series, fast=12, slow=26, signal=9):
    fast_ema = _ema(series, fast)
    slow_ema = _ema(series, slow)
    macd_line = fast_ema - slow_ema
    signal_line = _ema(macd_line, signal)
    hist_line = macd_line - signal_line
    return macd_line, signal_line, hist_line

# =====================================================================
# 1. EX Scanner Module (极值反转矩阵)
# =====================================================================
def add_ex_signals(df):
    out = df.copy()
    c, o, h, l = out['close'], out['open'], out['high'], out['low']

    sma10, sma20, sma50 = _sma(c, 10), _sma(c, 20), _sma(c, 50)
    macdLine, signalLine, _ = _macd(c)

    isNewLow = l < l.shift(1)
    is_red = c < o
    body = np.abs(c - o)
    lower_wick = np.minimum(o, c) - l

    isMomAccelDown = (sma10 - sma20) >= (sma20 - sma50)

    # EX-1 Buy
    ex1_delta = sma10 - l
    ex1_base = (c < sma10) & (sma10 < sma20) & (sma20 < sma50)
    ex1_candle = isNewLow & (ex1_delta > ex1_delta.shift(1)) & (o < c.shift(1))
    ex1_shape = is_red & (lower_wick > body * 0.6) & (l < sma10 * 0.85)
    out['buy_EX1'] = ex1_base & ex1_candle & ex1_shape & isMomAccelDown

    # EX-2 Buy
    ex2_base = (c < sma10) & (sma10 < sma20)
    ex2_shape = isNewLow & (l < sma10 * 0.85) & (lower_wick > body * 0.6)
    out['buy_EX2'] = ex2_base & ex2_shape & isMomAccelDown

    # EX-4 Buy
    ex4_macd = (macdLine < 0) & (macdLine > signalLine)
    ex4_cross = (c > sma20) & (c.shift(1) < sma20.shift(1))
    ex4_range = (c < sma20 * 1.02) & (sma20 >= sma20.shift(1))
    out['buy_EX4'] = ex4_macd & ex4_cross & ex4_range & isMomAccelDown

    return out

print("=" * 65)
print("  Scanner- CELL 7 - EX SCANNER (SGX + S&P 500)")
print("=" * 65)


# Load scanner data from memory, /content, or the newest matching files in Downloads.
if "ensure_scanner_data_loaded" in globals():
    ensure_scanner_data_loaded()
else:
    print("Run the setup/progress cell first so scanners can fall back to /content or Downloads files.")

sgx_ex_matches, sp500_ex_matches = [], []

if 'sgx_stock_data' in globals():
    for t, df in tqdm(sgx_stock_data.items(), desc="Scanning SGX (EX)"):
        if len(df) < 50: continue
        try:
            res = add_ex_signals(df).iloc[-1]
            if res.get('buy_EX1') or res.get('buy_EX2') or res.get('buy_EX4'):
                symbol = t.replace('.SI', '')
                sgx_ex_matches.append(symbol)
                signals = [s for s in ['EX1', 'EX2', 'EX4'] if bool(res.get('buy_' + s))]
                info = globals().get('sgx_ticker_info', {}).get(t, {})
                if 'record_scanner_pick' in globals():
                    record_scanner_pick(symbol, 'SGX', 'Cell 7 - EX', ', '.join(signals), info.get('name', ''), info.get('market', ''), 'Extreme reversal: ' + ', '.join(signals), round(float(df['close'].iloc[-1]), 4), df.index[-1].strftime('%Y-%m-%d'))
        except Exception: pass

    print(f"SGX EX Matches: {len(sgx_ex_matches)}")

if 'sp500_stock_data' in globals():
    for t, df in tqdm(sp500_stock_data.items(), desc="Scanning S&P 500 (EX)"):
        if len(df) < 50: continue
        try:
            res = add_ex_signals(df).iloc[-1]
            if res.get('buy_EX1') or res.get('buy_EX2') or res.get('buy_EX4'):
                sp500_ex_matches.append(t)
                signals = [s for s in ['EX1', 'EX2', 'EX4'] if bool(res.get('buy_' + s))]
                info = globals().get('sp500_ticker_info', {}).get(t, {})
                if 'record_scanner_pick' in globals():
                    record_scanner_pick(t, 'S&P 500', 'Cell 7 - EX', ', '.join(signals), info.get('name', ''), info.get('sector', ''), 'Extreme reversal: ' + ', '.join(signals), round(float(df['close'].iloc[-1]), 4), df.index[-1].strftime('%Y-%m-%d'))
        except Exception: pass

    print(f"S&P 500 EX Matches: {len(sp500_ex_matches)}")

print("Results kept in memory for Cell 11; no separate watchlist file exported from this scanner.")


  Scanner- CELL 7 - EX SCANNER (SGX + S&P 500)


Scanning SGX (EX):   0%|          | 0/122 [00:00<?, ?it/s]

SGX EX Matches: 0


Scanning S&P 500 (EX):   0%|          | 0/503 [00:00<?, ?it/s]

S&P 500 EX Matches: 1


## Scanner- Cell 8 — MS Scanner (Master Screener PRO)
Scans for MS-1 (Momentum Rebound) and MS-3 (20SMA Bounce) buy signals across SGX and S&P 500.
If historical data is missing from memory, the scanner will use only fresh saved CSVs from `/content`, Google Drive `ScannerData`, or Downloads; stale files are ignored.


In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

def _sma(series, length): return series.rolling(window=length, min_periods=1).mean()
def _ema(series, length): return series.ewm(span=length, adjust=False).mean()
def _rsi(series, length=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=length, min_periods=1).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=length, min_periods=1).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))
def _macd(series, fast=12, slow=26, signal=9):
    fast_ema = _ema(series, fast); slow_ema = _ema(series, slow)
    macd_line = fast_ema - slow_ema; signal_line = _ema(macd_line, signal)
    return macd_line, signal_line, macd_line - signal_line

# =====================================================================
# 2. MS Scanner Module (Master Screener PRO)
# =====================================================================
def add_ms_signals(df, rsi_os_level=40):
    out = df.copy()
    c, o, h, l, v = out['close'], out['open'], out['high'], out['low'], out['volume']

    sma10, sma20, sma50 = _sma(c, 10), _sma(c, 20), _sma(c, 50)
    vol_sma = _sma(v, 20)
    rel_vol = v / vol_sma.replace(0, np.nan)
    macdLine, signalLine, histLine = _macd(c)
    rsi_val = _rsi(c); rsi_sma = _sma(rsi_val, 14)

    is_green = c > o
    upper_wick = h - np.maximum(o, c)
    body = np.abs(c - o)

    # MS-1
    ms1_trend = (c < sma10) & (sma10 < sma20)
    ms1_candle = (rel_vol > 1.5) | (is_green & (upper_wick < body * 0.5))
    ms1_rsi = (rsi_val < rsi_os_level) & (np.abs(rsi_val - rsi_sma) <= 9)
    ms1_macd = (histLine > histLine.shift(1)) & (histLine >= 0)
    ms1_price = (o - o.shift(1)) - (c - c.shift(1)) < 0
    ms1_low = l <= l.shift(1) * 1.005
    out['buy_MS1'] = ms1_trend & ms1_price & ms1_macd & ms1_candle & ms1_rsi & ms1_low

    # MS-3
    ms3_trend = (sma20 > sma50) & (sma50 >= sma50.shift(1))
    ms3_setup = (c.shift(1) <= sma20.shift(1) * 1.01) & (c.shift(1) >= sma50.shift(1))
    ms3_candle = (c > sma20) & is_green & (c > o.shift(1)) & (upper_wick <= body) & (c > sma10 * 0.9)
    ms3_vol = rel_vol > 1.1
    ms3_rsi = rsi_sma >= rsi_sma.shift(3)
    out['buy_MS3'] = ms3_trend & ms3_setup & ms3_candle & ms3_vol & ms3_rsi

    return out

print("=" * 65)
print("  Scanner- CELL 8 - MS SCANNER (SGX + S&P 500)")
print("=" * 65)


# Load scanner data from memory, /content, or the newest matching files in Downloads.
if "ensure_scanner_data_loaded" in globals():
    ensure_scanner_data_loaded()
else:
    print("Run the setup/progress cell first so scanners can fall back to /content or Downloads files.")

sgx_ms_matches, sp500_ms_matches = [], []

if 'sgx_stock_data' in globals():
    for t, df in tqdm(sgx_stock_data.items(), desc="Scanning SGX (MS)"):
        if len(df) < 50: continue
        try:
            res = add_ms_signals(df).iloc[-1]
            if res.get('buy_MS1') or res.get('buy_MS3'):
                symbol = t.replace('.SI', '')
                sgx_ms_matches.append(symbol)
                signals = [s for s in ['MS1', 'MS3'] if bool(res.get('buy_' + s))]
                info = globals().get('sgx_ticker_info', {}).get(t, {})
                if 'record_scanner_pick' in globals():
                    record_scanner_pick(symbol, 'SGX', 'Cell 8 - MS', ', '.join(signals), info.get('name', ''), info.get('market', ''), 'Master screener: ' + ', '.join(signals), round(float(df['close'].iloc[-1]), 4), df.index[-1].strftime('%Y-%m-%d'))
        except Exception: pass
    print(f"SGX MS Matches: {len(sgx_ms_matches)}")

if 'sp500_stock_data' in globals():
    for t, df in tqdm(sp500_stock_data.items(), desc="Scanning S&P 500 (MS)"):
        if len(df) < 50: continue
        try:
            res = add_ms_signals(df).iloc[-1]
            if res.get('buy_MS1') or res.get('buy_MS3'):
                sp500_ms_matches.append(t)
                signals = [s for s in ['MS1', 'MS3'] if bool(res.get('buy_' + s))]
                info = globals().get('sp500_ticker_info', {}).get(t, {})
                if 'record_scanner_pick' in globals():
                    record_scanner_pick(t, 'S&P 500', 'Cell 8 - MS', ', '.join(signals), info.get('name', ''), info.get('sector', ''), 'Master screener: ' + ', '.join(signals), round(float(df['close'].iloc[-1]), 4), df.index[-1].strftime('%Y-%m-%d'))
        except Exception: pass
    print(f"S&P 500 MS Matches: {len(sp500_ms_matches)}")

print("Results kept in memory for Cell 11; no separate watchlist file exported from this scanner.")


  Scanner- CELL 8 - MS SCANNER (SGX + S&P 500)


Scanning SGX (MS):   0%|          | 0/122 [00:00<?, ?it/s]

SGX MS Matches: 0


Scanning S&P 500 (MS):   0%|          | 0/503 [00:00<?, ?it/s]

S&P 500 MS Matches: 0


## Scanner- Cell 9 — AM Scanner (Annual Macro)
Scans for AM-12 (180-day High) and AM-13 (250-day High) breakouts across SGX and S&P 500.
If historical data is missing from memory, the scanner will use only fresh saved CSVs from `/content`, Google Drive `ScannerData`, or Downloads; stale files are ignored.


In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

def _sma(series, length): return series.rolling(window=length, min_periods=1).mean()

# =====================================================================
# 3. AM Scanner Module (Annual Macro)
# =====================================================================
def add_am_signals(df):
    out = df.copy()
    c, o, h = out['close'], out['open'], out['high']
    is_green = c > o
    sma20, sma50 = _sma(c, 20), _sma(c, 50)
    body = np.abs(c - o)
    upper_wick = h - np.maximum(o, c)

    # AM-12: 180-Day High Breakout
    am12_prev = h.shift(1).rolling(180).max()
    am12_hit = is_green & (h > am12_prev)
    am12_str = am12_prev >= (h * 0.95)
    am12_wick = upper_wick <= (body * 0.10)
    am12_ma = (c > sma20) & (sma20 > sma50)
    out['buy_AM12'] = am12_hit & am12_str & am12_wick & am12_ma

    # AM-13: 250-Day Yearly High Breakout
    am13_prev = h.shift(1).rolling(250).max()
    am13_hit = is_green & (h > am13_prev)
    am13_str = am13_prev >= (h * 0.95)
    out['buy_AM13'] = am13_hit & am13_str & am12_wick & am12_ma

    return out

print("=" * 65)
print("  Scanner- CELL 9 - AM SCANNER (SGX + S&P 500)")
print("=" * 65)


# Load scanner data from memory, /content, or the newest matching files in Downloads.
if "ensure_scanner_data_loaded" in globals():
    ensure_scanner_data_loaded()
else:
    print("Run the setup/progress cell first so scanners can fall back to /content or Downloads files.")

sgx_am_matches, sp500_am_matches = [], []

if 'sgx_stock_data' in globals():
    for t, df in tqdm(sgx_stock_data.items(), desc="Scanning SGX (AM)"):
        if len(df) < 260: continue
        try:
            res = add_am_signals(df).iloc[-1]
            if res.get('buy_AM12') or res.get('buy_AM13'):
                symbol = t.replace('.SI', '')
                sgx_am_matches.append(symbol)
                signals = [s for s in ['AM12', 'AM13'] if bool(res.get('buy_' + s))]
                info = globals().get('sgx_ticker_info', {}).get(t, {})
                if 'record_scanner_pick' in globals():
                    record_scanner_pick(symbol, 'SGX', 'Cell 9 - AM', ', '.join(signals), info.get('name', ''), info.get('market', ''), 'Annual macro breakout: ' + ', '.join(signals), round(float(df['close'].iloc[-1]), 4), df.index[-1].strftime('%Y-%m-%d'))
        except Exception: pass
    print(f"SGX AM Matches: {len(sgx_am_matches)}")

if 'sp500_stock_data' in globals():
    for t, df in tqdm(sp500_stock_data.items(), desc="Scanning S&P 500 (AM)"):
        if len(df) < 260: continue
        try:
            res = add_am_signals(df).iloc[-1]
            if res.get('buy_AM12') or res.get('buy_AM13'):
                sp500_am_matches.append(t)
                signals = [s for s in ['AM12', 'AM13'] if bool(res.get('buy_' + s))]
                info = globals().get('sp500_ticker_info', {}).get(t, {})
                if 'record_scanner_pick' in globals():
                    record_scanner_pick(t, 'S&P 500', 'Cell 9 - AM', ', '.join(signals), info.get('name', ''), info.get('sector', ''), 'Annual macro breakout: ' + ', '.join(signals), round(float(df['close'].iloc[-1]), 4), df.index[-1].strftime('%Y-%m-%d'))
        except Exception: pass
    print(f"S&P 500 AM Matches: {len(sp500_am_matches)}")

print("Results kept in memory for Cell 11; no separate watchlist file exported from this scanner.")


  Scanner- CELL 9 - AM SCANNER (SGX + S&P 500)


Scanning SGX (AM):   0%|          | 0/122 [00:00<?, ?it/s]

SGX AM Matches: 0


Scanning S&P 500 (AM):   0%|          | 0/503 [00:00<?, ?it/s]

S&P 500 AM Matches: 0


## Scanner - Cell 10 - Breakout buy
If historical data is missing from memory, the scanner will use only fresh saved CSVs from `/content`, Google Drive `ScannerData`, or Downloads; stale files are ignored.


In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

def _sma(series, length): return series.rolling(window=length, min_periods=1).mean()

# =====================================================================
# 4. RB Scanner Module (Resistance Breakout Buy)
# =====================================================================
def add_rb_signals(df, lb=10, clustPct=1.0, boMin=2.0, boMax=10.0):
    out = df.copy()
    c, o, h, l = out['close'], out['open'], out['high'], out['low']

    # SMAs
    sma10, sma20, sma50 = _sma(c, 10), _sma(c, 20), _sma(c, 50)

    # Signal arrays
    buy_sig = np.zeros(len(out), dtype=bool)

    # We need at least 'lb' past bars to compute
    for i in range(lb, len(out)):
        # Resistance lookback (excluding current bar i)
        # In Pandas, .iloc[i-lb:i] gets the previous 'lb' bars
        past_highs = h.iloc[i-lb:i]
        past_closes = c.iloc[i-lb:i]

        resLine = past_highs.max()

        # Count cluster touches AND find leftmost pivot
        clustCnt = 0
        leftmostI = 0

        # Iterate backwards to match Pine Script's lookback indexing (1 to lb)
        for j in range(1, lb + 1):
            past_h = h.iloc[i - j]
            if past_h >= resLine * (1.0 - clustPct / 100.0):
                clustCnt += 1
                if j > leftmostI:
                    leftmostI = j

        # Conditions
        c1_trend    = (sma10.iloc[i] > sma20.iloc[i]) and (sma20.iloc[i] > sma50.iloc[i])
        c2_candle   = (c.iloc[i] > o.iloc[i]) and (o.iloc[i] > sma20.iloc[i])
        c3_topClose = c.iloc[i] > past_closes.max()
        c4_cluster  = clustCnt >= 2
        c5_breakout = (c.iloc[i] >= resLine * (1.0 + boMin / 100.0)) and (c.iloc[i] <= resLine * (1.0 + boMax / 100.0))
        c6_gap      = leftmostI >= 3

        signal = c1_trend and c2_candle and c3_topClose and c4_cluster and c5_breakout and c6_gap
        buy_sig[i] = signal

    out['buy_RB'] = buy_sig
    return out

print("=" * 65)
print("  Scanner- CELL 9.5 - RB SCANNER (SGX + S&P 500)")
print("=" * 65)


# Load scanner data from memory, /content, or the newest matching files in Downloads.
if "ensure_scanner_data_loaded" in globals():
    ensure_scanner_data_loaded()
else:
    print("Run the setup/progress cell first so scanners can fall back to /content or Downloads files.")

sgx_rb_matches, sp500_rb_matches = [], []

if 'sgx_stock_data' in globals():
    for t, df in tqdm(sgx_stock_data.items(), desc="Scanning SGX (RB)"):
        if len(df) < 60: continue
        try:
            res = add_rb_signals(df).iloc[-1]
            if res.get('buy_RB'):
                symbol = t.replace('.SI', '')
                sgx_rb_matches.append(symbol)
                info = globals().get('sgx_ticker_info', {}).get(t, {})
                if 'record_scanner_pick' in globals():
                    record_scanner_pick(symbol, 'SGX', 'Cell 10 - RB', 'RB', info.get('name', ''), info.get('market', ''), 'Resistance breakout buy', round(float(df['close'].iloc[-1]), 4), df.index[-1].strftime('%Y-%m-%d'))
        except Exception: pass
    print(f"SGX RB Matches: {len(sgx_rb_matches)}")

if 'sp500_stock_data' in globals():
    for t, df in tqdm(sp500_stock_data.items(), desc="Scanning S&P 500 (RB)"):
        if len(df) < 60: continue
        try:
            res = add_rb_signals(df).iloc[-1]
            if res.get('buy_RB'):
                sp500_rb_matches.append(t)
                info = globals().get('sp500_ticker_info', {}).get(t, {})
                if 'record_scanner_pick' in globals():
                    record_scanner_pick(t, 'S&P 500', 'Cell 10 - RB', 'RB', info.get('name', ''), info.get('sector', ''), 'Resistance breakout buy', round(float(df['close'].iloc[-1]), 4), df.index[-1].strftime('%Y-%m-%d'))
        except Exception: pass
    print(f"S&P 500 RB Matches: {len(sp500_rb_matches)}")
print("Results kept in memory for Cell 11; no separate watchlist file exported from this scanner.")


  Scanner- CELL 9.5 - RB SCANNER (SGX + S&P 500)


Scanning SGX (RB):   0%|          | 0/122 [00:00<?, ?it/s]

SGX RB Matches: 1


Scanning S&P 500 (RB):   0%|          | 0/503 [00:00<?, ?it/s]

S&P 500 RB Matches: 0


# *** COMBINE & DOWNLOAD
Cell 11 consolidates scanner results from Cells 5 to 10, deduplicates symbols, and creates two files: a detailed CSV explaining why each stock was picked, plus a TradingView watchlist.


## Download Module - Cell 11

In [ ]:
import os
import pandas as pd

print("=" * 65)
print("  CELL 11 - CONSOLIDATE RESULTS")
print("=" * 65)


# Near-final history backup: only empties Drive ScannerData if fresh online history was created this session.
if "backup_fresh_history_to_drive_near_final_step" in globals():
    backup_fresh_history_to_drive_near_final_step()

# Collect results that were sent directly by Cells 6 to 10.
rows = list(globals().get("scanner_results", []))

# Backward-compatible fallback for Cell 5 if it created old-style match lists.
def _fallback_rows(match_rows, market, scanner):
    out = []
    for item in match_rows:
        if isinstance(item, dict):
            symbol = item.get("symbol") or item.get("ticker") or item.get("tv_symbol")
            if not symbol:
                continue
            out.append({
                "tv_symbol": f"SGX:{symbol}" if market == "SGX" and not str(symbol).startswith("SGX:") else str(symbol),
                "symbol": str(symbol).replace("SGX:", ""),
                "market": market,
                "name": item.get("name", ""),
                "sector": item.get("sector", item.get("market", "")),
                "scanner": scanner,
                "signal": item.get("signal", scanner),
                "reason": item.get("reason", item.get("signal", scanner)),
                "last_close": item.get("last_close"),
                "last_date": item.get("last_date"),
            })
        else:
            symbol = str(item).replace("SGX:", "")
            out.append({
                "tv_symbol": f"SGX:{symbol}" if market == "SGX" else symbol,
                "symbol": symbol,
                "market": market,
                "name": "",
                "sector": "",
                "scanner": scanner,
                "signal": scanner,
                "reason": scanner,
                "last_close": None,
                "last_date": None,
            })
    return out

rows.extend(_fallback_rows(globals().get("sgx_matched", []), "SGX", "Cell 5 - Stock Scanner"))
rows.extend(_fallback_rows(globals().get("sp500_matched", []), "S&P 500", "Cell 5 - Stock Scanner"))

if not rows:
    print("No scanner results found. Run scanner Cells 5 to 10 first, then run Cell 11 again.")
else:
    raw = pd.DataFrame(rows)
    for col in ["tv_symbol", "symbol", "market", "name", "sector", "scanner", "signal", "reason", "last_close", "last_date"]:
        if col not in raw.columns:
            raw[col] = ""

    raw["tv_symbol"] = raw["tv_symbol"].astype(str).str.strip()
    raw = raw[raw["tv_symbol"] != ""]

    def _join_unique(values):
        seen, output = set(), []
        for value in values:
            if pd.isna(value) or value == "":
                continue
            text = str(value)
            if text not in seen:
                seen.add(text)
                output.append(text)
        return "; ".join(output)

    consolidated = (
        raw.groupby("tv_symbol", as_index=False)
           .agg({
               "symbol": "first",
               "market": "first",
               "name": _join_unique,
               "sector": _join_unique,
               "scanner": _join_unique,
               "signal": _join_unique,
               "reason": _join_unique,
               "last_close": "last",
               "last_date": "last",
           })
    )
    consolidated["picked_by_count"] = consolidated["scanner"].apply(lambda x: len([v for v in str(x).split("; ") if v]))
    consolidated = consolidated[[
        "tv_symbol", "symbol", "market", "name", "sector", "picked_by_count",
        "scanner", "signal", "reason", "last_close", "last_date"
    ]].sort_values(["market", "symbol"])

    detail_file = "/content/Consolidated_Scanner_Results.csv"
    watchlist_file = "/content/TradingView_Watchlist.txt"

    consolidated.to_csv(detail_file, index=False)
    with open(watchlist_file, "w", encoding="utf-8") as f:
        f.write("\n".join(consolidated["tv_symbol"].tolist()))
        f.write("\n")

    print(f"Consolidated {len(raw)} scanner hits into {len(consolidated)} unique TradingView symbols.")
    print(f"Detailed CSV: {detail_file}")
    print(f"TradingView watchlist: {watchlist_file}")

    if "download_to_local" in globals():
        download_to_local([detail_file, watchlist_file])
    else:
        try:
            from google.colab import files
            files.download(detail_file)
            files.download(watchlist_file)
        except Exception as e:
            print(f"Could not automatically trigger download: {e}")


  Wrapping up - CELL 10 - COMBINE & DOWNLOAD AS ZIP
✅ Successfully combined 38 unique tickers from all active scans.

Packing files into ZIP...
  Added: Master_TradingView_Watchlist.txt
  Added: sgx_historical.csv
  Added: sp500_historical.csv
  Added: sgx_scan_results.csv
  Added: sp500_scan_results.csv
  Added: sgx_ab20exp_results.csv
  Added: sp500_ab20exp_results.csv

✅ All available results packed into /content/Scanner_Results.zip
Triggering single direct download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>